# Step 8 — Environmental Justice Overlay

**Objective:** Test whether climate-amplified outage risk disproportionately affects disadvantaged communities.

**Data source:** EPA EJScreen 2024 — https://www.epa.gov/ejscreen or https://zenodo.org/records/14767363

**Tests:**
1. Outage rates by EJ burden quartile
2. Regression: outage severity controlling for weather in high-EJ counties
3. Interaction: compound weather x EJ burden
4. Forward projection: high-EJ + high-heat counties by 2050

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import PROCESSED, ERCOT_FIPS
from src.data.ejscreen import build_ercot_ejscreen
from src.analysis.panel_regression import run_panel_ols, summarise_results


## 8.1 Build county-level EJ indicators

In [ ]:
try:
    ej_ercot = build_ercot_ejscreen()
    print(f'ERCOT EJScreen counties: {len(ej_ercot)}')
    print(ej_ercot.head())
except FileNotFoundError as e:
    print(f'EJScreen not found:\n{e}')
    ej_ercot = None


## 8.2 Merge with historical outage panel

In [ ]:
ej_merged = None
if PROCESSED['merged_ercot'].exists() and ej_ercot is not None:
    merged = pd.read_csv(PROCESSED['merged_ercot'], parse_dates=['date'])
    merged['fips'] = merged['fips'].astype(str).str.zfill(5)
    ej_merged = merged.merge(ej_ercot.reset_index(), on='fips', how='left')
    print(f'Merged shape: {ej_merged.shape}')
    print(f'EJ merge coverage: {ej_merged["ej_index"].notna().mean():.1%}')
else:
    print('Merged panel or EJScreen missing.')


## 8.3 Outage rates by EJ burden quartile

In [ ]:
if ej_merged is not None and 'ej_index' in ej_merged.columns:
    ej_merged['ej_quartile'] = pd.qcut(ej_merged['ej_index'].rank(method='first'), 4,
                                        labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])
    q_rates = ej_merged.groupby('ej_quartile')['outage_event_flag'].mean()
    print('Outage event rate by EJ burden quartile:')
    print(q_rates)

    fig, ax = plt.subplots(figsize=(8, 5))
    q_rates.plot(kind='bar', ax=ax, color='darkred', edgecolor='white')
    ax.set_title('Outage event rate by EJ burden quartile — ERCOT')
    ax.set_ylabel('Outage event rate')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig('../data/processed/ercot_ej_outage_rates.png', dpi=150)
    plt.show()


## 8.4 Regression: outage severity in high-EJ counties

In [ ]:
if ej_merged is not None:
    model_cols = ['fips', 'date', 'total_customer_hours',
                  'heatwave_day', 'compound_triple', 'high_ej_burden']
    avail = [c for c in model_cols if c in ej_merged.columns]
    df_model = ej_merged[avail].dropna().set_index(['fips', 'date'])
    if 'high_ej_burden' in df_model.columns:
        result_ej = run_panel_ols(df_model, outcome='total_customer_hours',
                                  weather_vars=['heatwave_day', 'compound_triple'],
                                  controls=['high_ej_burden'])
        print(result_ej.summary)


## 8.5 Interaction: compound weather x EJ burden

In [ ]:
if ej_merged is not None and 'compound_triple' in ej_merged.columns and 'high_ej_burden' in ej_merged.columns:
    ej_merged['compound_x_ej'] = ej_merged['compound_triple'] * ej_merged['high_ej_burden']
    df_i = ej_merged[['fips', 'date', 'total_customer_hours',
                       'compound_triple', 'high_ej_burden', 'compound_x_ej']].dropna()
    result_int = run_panel_ols(df_i.set_index(['fips', 'date']),
                               outcome='total_customer_hours',
                               weather_vars=['compound_triple', 'high_ej_burden', 'compound_x_ej'])
    print(result_int.summary)
    print('\nKey: compound_x_ej tests whether compound outage effect is LARGER in high-EJ counties.')


## 8.6 Forward projection: high-EJ + high-heat double-exposure counties

In [ ]:
if ej_ercot is not None and PROCESSED['loca2_ercot'].exists():
    loca2 = pd.read_csv(PROCESSED['loca2_ercot'])
    mid = loca2[(loca2['scenario'] == 'ssp585') & (loca2['period_label'] == 'mid')]
    ej_loca2 = ej_ercot.reset_index().merge(mid[['fips', 'SU_delta']].dropna(), on='fips', how='inner')
    ej_loca2['both_high'] = (
        (ej_loca2.get('high_ej_burden', 0) == 1) &
        (ej_loca2['SU_delta'] >= ej_loca2['SU_delta'].quantile(0.75))
    ).astype(int)
    n_both = ej_loca2['both_high'].sum()
    print(f'Counties with HIGH EJ burden AND HIGH projected heat increase: {n_both}')
    print(ej_loca2[ej_loca2['both_high'] == 1][['fips', 'ej_index', 'SU_delta']].head(10))
